In [49]:
import pandas as pd
import json
df = pd.read_csv('data/raw/resources.csv')

#pd.set_option("display.max_rows", None)
df.columns = ["ResourceId", "OrgId", "Service","Region","CreatedDate","State","Tags"]
#Eliminamos los primeros 4 carácteres de las dos columnas ya que son repetitivos
df['ResourceId'] = df['ResourceId'].str[4:]
df['OrgId'] = df['OrgId'].str[4:]
# Cuenta los NaN en la columna 'tags
conteo_nan = df['Tags'].isna().sum()

print(f'Cantidad de valores NaN: {conteo_nan}')
total_filas = len(df)
print(f'Total de filas: {total_filas}')

print(df.dtypes)

#CreatedDate tomada como Object(string)
#pasamos a DateTime
df['CreatedDate'] = pd.to_datetime(df['CreatedDate'], errors='coerce')
print(df.dtypes)
#vemos el cambio
#Verifico si existen duplicados de usuarios o de mail
resource_duplicados = df['ResourceId'].duplicated().sum()
org_duplicadas = df['OrgId'].duplicated().sum()

print(f"resources duplicados: {resource_duplicados}")
print(f"orgs duplicados: {org_duplicadas}")

# Resumen ordenado de las organizaciones con más recursos
resumen = df.groupby('OrgId').agg(
    total_recursos=('ResourceId', 'count'),
    servicios_distintos=('Service', 'nunique'),
    primer_recurso=('CreatedDate', 'min'),
    ultimo_recurso=('CreatedDate', 'max')
).reset_index().sort_values(by='total_recursos', ascending=False)

display(resumen.head(10))
#Verificamos valores unicos en columnas
print(df['Service'].nunique(), df['Service'].unique())
print(df['Region'].nunique(), df['Region'].unique())
print(df['State'].nunique(), df['State'].unique())
#Notamos que en las columnas hay 6/7/3 valores distintos en 400 lineas

# Convertimos en tipo categórica
df['Service'] = df['Service'].astype('category')
df['Region'] = df['Region'].astype('category')
df['State'] = df['State'].astype('category')
#Verificamos
print(df[['Service', 'Region', 'State']].dtypes)

#Remplazamos valores NaN de Tags

df['Tags'] = df['Tags'].fillna('[]')
#Revisamos
df['Tags']

# Convertir cada celda de texto a lista nativa
df['Tags'] = df['Tags'].apply(json.loads)

# 2. Función que convierte ['env:prod', 'pii:true'] -> {'env': 'prod', 'pii': 'true'}
def parsear_tags_a_dict(lista_tags):
    diccionario = {}
    for tag in lista_tags:
        if ':' in tag:
            clave, valor = tag.split(':', 1)
            diccionario[clave] = valor
    return diccionario

df_categorias = df['Tags'].apply(parsear_tags_a_dict).apply(pd.Series)
df = pd.concat([df, df_categorias], axis=1)
#En caso de la división de columnas, cuando algunos valores sean NaN y la columna sea booleana, asignamos true or false
df['pii'] = df['pii'].fillna(False).replace({'true': True})
df['backup'] = df['backup'].fillna(False).replace({'daily': True})
#EN string, asignamos otros valores
df['env'] = df['env'].fillna('Sin Especificar')
df['costcenter'] = df['costcenter'].fillna('Sin Asignar')
df['team'] = df['team'].fillna('Sin Asignar')
#por ultimo borramos la columna tags

df=df.drop(columns=['Tags'])

df.to_csv('data/clean/resources_limpio.csv', index=False, encoding='utf-8')

Cantidad de valores NaN: 83
Total de filas: 400
ResourceId     object
OrgId          object
Service        object
Region         object
CreatedDate    object
State          object
Tags           object
dtype: object
ResourceId             object
OrgId                  object
Service                object
Region                 object
CreatedDate    datetime64[ns]
State                  object
Tags                   object
dtype: object
resources duplicados: 0
orgs duplicados: 320


,OrgId,total_recursos,servicios_distintos,primer_recurso,ultimo_recurso
42,kdgigatj,11,5,2025-05-12,2025-08-10
9,53lc58dr,10,6,2025-05-10,2025-08-12
53,pbhsahxt,10,6,2025-05-10,2025-08-14
27,dhylurtp,9,4,2025-05-24,2025-07-23
22,chj755nf,9,5,2025-05-19,2025-08-20
50,okep7y6w,8,5,2025-05-18,2025-08-12
47,nam148p0,8,4,2025-05-08,2025-08-04
43,ktakpuxq,8,4,2025-05-13,2025-07-06
23,cvs4f8cg,8,6,2025-05-05,2025-08-10
3,1swjckjl,7,2,2025-05-15,2025-07-28


6 ['compute' 'database' 'storage' 'networking' 'analytics' 'genai']
7 ['sa-east' 'ap-south' 'eu-central' 'us-west' 'ap-northeast' 'eu-west'
 'us-east']
3 ['running' 'stopped' 'terminated']
Service    category
Region     category
State      category
dtype: object


/tmp/ipykernel_2855/972032174.py:74: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['pii'] = df['pii'].fillna(False).replace({'true': True})
/tmp/ipykernel_2855/972032174.py:75: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['backup'] = df['backup'].fillna(False).replace({'daily': True})
